In [ ]:
# import libraries
import pandas as pd
import ee
import geemap

## Connect to GOOGLE EARTH ENGINE

In [ ]:
# Authenticate GEE
ee.Authenticate()

In [ ]:
# Initialize GEE
ee.Initialize()

print("Google Earth Engine initialized successfully!")

In [ ]:
vis_params_Fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}
vis_params_aio =  {"fillColor":"", "color": "red"}

In [ ]:
# Boundary Data From FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGAs boundaries



#dataset = dataset.style(styleParams)

boundary_Map = geemap.Map(center=(7.0, 8.0), zoom=10)
boundary_Map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_Map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_Map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_Map

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja"))
print(nga_l0.getInfo())
#get geometry of Abuja Boundary Feature Collection
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

#Create a map to visualize Abuja Boundary
aoi_map = geemap.Map(center=(7.0, 8.0), zoom=10)

# 7. Add layer (Fixed capital 'L' in addLayer)
aoi_map.addLayer(fct_l0, vis_params_aio, 'Abuja Boundary')
aoi_map


## Explore image operations
### AOI, SCL Cloud Masking, & Median Composite

In [ ]:
# Cloud masking function using SCL (Scene Classification Layer)
def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Keep clear land (4), vegetation (5), water (6), unclassified (7)
    mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return image.updateMask(mask)

# Download and process Sentinel-2 collection over Abuja
def get_sentinel2_composite(aoi, start_date='2022-01-01', end_date='2022-01-31'):
    s2_img_col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
    )
    return s2_img_col

# Call the function to create s2_img_col
s2_img_col = get_sentinel2_composite(aoi)

# Collection Properties
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}")

# Get the first image in the collection and print its properties
first_s2_img = s2_img_col.first()
print(f"First image in S2 collection: {first_s2_img.getInfo()}")

# Bands in S2 image
print(f"Bands in S2 image: {first_s2_img.bandNames().getInfo()}")

# Select just "B4"
red_band_s2 = first_s2_img.select('B4')
print(f"Red band (B4) in S2 image: {red_band_s2.bandNames().getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_map.addLayer(first_s2_img, vis_params_s2, 'First S2 Image')
s2_map

In [ ]:
# Mosaic the S2 collection to create a single image
s2_img_median = s2_img_col.median()
print(f"Median of S2 collection: {s2_img_median.getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_median_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_median_map.addLayer(s2_img_median, vis_params_s2, 'S2 Median Image')
s2_median_map

In [ ]:
# Mosaic the S2 collection to create a single image
s2_img_mean = s2_img_col.mean()
print(f"Mean of S2 collection: {s2_img_mean.getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_mean_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_mean_map.addLayer(s2_img_mean, vis_params_s2, 'S2 Mean Image')
s2_mean_map

Road Data & Distance to road using GEE

In [ ]:
# Loading the GRIP4 Africa Roads dataset
roads_africa = ee.FeatureCollection(
    "projects/sat-io/open-datasets/GRIP4/Africa"
)

# Extracting roads within the study area
roads_aoi = roads_africa.filterBounds(aoi_bbox)

# Converting road vectors to raster
roads_raster = (
    ee.Image()
    .float()
    .paint(roads_aoi, 1)
    .clip(aoi)
)

# Computing Euclidean Distance to Roads (meters)
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_roads")
    .clip(aoi)
)

# Print maximum distance value
max_distance = distance_to_roads.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=aoi,
    scale=1000,
    maxPixels=1e9
)

print("Maximum Distance to Road (m):")
print(max_distance.getInfo())


In [ ]:
# Visualization Parameters

vis_params_roads_vector = {
    "color": "red"
}

vis_params_roads_raster = {
    "min": 0,
    "max": 1,
    "palette": ["white", "black"]
}

vis_params_dist_road = {
    "min": 0,
    "max": 5000,
    "palette": [
        "blue",
        "cyan",
        "green",
        "yellow",
        "orange",
        "red"
    ]
}


In [ ]:
# Layers display

aoi_map.addLayer(
    roads_raster,
    vis_params_roads_raster,
    "Road Raster"
)

aoi_map.addLayer(
    distance_to_roads.select("dist_to_roads"),
    vis_params_dist_road,
    "Distance to Road"
)

aoi_map.addLayer(
    roads_aoi,
    vis_params_roads_vector,
    "Road Vector"
)

thematic_map_1 = aoi_map

# Display Map
thematic_map_1

In [ ]:
# Nighttime Light Data using GEE

# Function to compute annual mean nighttime lights
def get_nighttime_lights(year, study_extent):
    """
    Returns annual mean VIIRS Nighttime Lights (avg_rad)
    for a given year and study area.
    """
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    composite = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
        .filterDate(start, end)
        .select("avg_rad")
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )

    return composite


# Generate Nighttime Lights Images

ntl_2015 = get_nighttime_lights(2015, aoi)
ntl_2020 = get_nighttime_lights(2020, aoi)


# Visualization Parameters


vis_params_ntl = {
    "min": 0,
    "max": 60,
    "palette": [
        "black",
        "purple",
        "blue",
        "cyan",
        "green",
        "yellow",
        "orange",
        "red",
        "white"
    ]
}

# Create Map

thematic_map_2 = geemap.Map(center=[9.05, 7.49], zoom=10)

# Abuja Boundary
thematic_map_2.addLayer(fct_l0, {}, "Abuja Boundary")

# Nighttime Lights Layers
thematic_map_2.addLayer(
    ntl_2015,
    vis_params_ntl,
    "Nighttime Lights - 2015"
)

thematic_map_2.addLayer(
    ntl_2020,
    vis_params_ntl,
    "Nighttime Lights - 2020"
)

# Display map
thematic_map_2